# 02 — Clean & Validate

Take the raw lead data from notebook 01 and turn it into clean, analysis-ready tables.

**Pipeline**: `raw/` → `staging/` (cleaned) → `mart/` (aggregated + KPIs)

**Key constant**: `COST_PER_LEAD = £55` — the actual cost we pay per lead.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
MART_DIR = PROJECT_ROOT / "data" / "mart"

STAGING_DIR.mkdir(parents=True, exist_ok=True)
MART_DIR.mkdir(parents=True, exist_ok=True)

COST_PER_LEAD = 55

## 1 · Load Raw Data

In [2]:
leads_raw = pd.read_csv(RAW_DIR / "leads" / "insurance_leads.csv")
lead_spend = pd.read_csv(RAW_DIR / "spend" / "lead_spend.csv", parse_dates=["week_start"])

print(f"Leads: {leads_raw.shape}")
print(f"Spend: {lead_spend.shape}")
leads_raw.head()

Leads: (7485, 13)
Spend: (1404, 8)


,lead_id,lead_status,premium,age,gender,smoker,current_insurance,device_type,keyword,match_type,postcode,cover_for,verification_status
0,90809057,Contacted,0.0,31,Female,Yes,no,Desktop,private health insurance belfast,Exact,Bt13,Self,NaN
1,35622726,Contacted,0.0,35,Female,No,no,Smartphone,private medical insurance,Phrase,LU7,Self,NaN
2,29156003,Contacted,0.0,46,Male,No,yes private,Smartphone,private health insurance,Broad,E14,Self,NaN
3,26436791,No Contact,0.0,53,Female,No,no,Desktop,bupa health insurance,Exact,W2,Self + Partner,NaN
4,80270348,Contacted,0.0,35,Male,No,no,Smartphone,private healthcare prices,Exact,BH23,Self,NaN


In [3]:
print(leads_raw['verification_status'].value_counts())
print("-------------------------------")
print("not verified:", leads_raw['verification_status'].isna().sum())

verification_status
verified    1554
Name: count, dtype: int64
-------------------------------
not verified: 5931


## 2 · Clean & Standardize

Steps:
1. Clean text values (strip whitespace, normalize case)
2. Extract postcode area for geographic analysis
3. Group keywords into meaningful themes
4. Create target flags (converted, invalid, high-value)
5. Add age bands and synthetic dates

In [4]:
leads = leads_raw.copy()

# Standardize text values
leads["postcode"] = leads["postcode"].str.upper().str.strip()
leads["verification_status"] = leads["verification_status"].fillna("not_verified")
leads["gender"] = leads["gender"].str.strip()
leads["smoker"] = leads["smoker"].str.strip().str.lower()
leads["current_insurance"] = leads["current_insurance"].str.strip()
leads["device_type"] = leads["device_type"].str.strip()
leads["match_type"] = leads["match_type"].str.strip()

# Postcode area (letter prefix for geographic analysis)
leads["pc_area"] = leads["postcode"].str.extract(r"^([A-Z]{1,2})")[0]

print(f"Shape: {leads.shape}")
print(f"Columns: {list(leads.columns)}")
leads.head()

Shape: (7485, 14)
Columns: ['lead_id', 'lead_status', 'premium', 'age', 'gender', 'smoker', 'current_insurance', 'device_type', 'keyword', 'match_type', 'postcode', 'cover_for', 'verification_status', 'pc_area']


,lead_id,lead_status,premium,age,gender,smoker,current_insurance,device_type,keyword,match_type,postcode,cover_for,verification_status,pc_area
0,90809057,Contacted,0.0,31,Female,yes,no,Desktop,private health insurance belfast,Exact,BT13,Self,not_verified,BT
1,35622726,Contacted,0.0,35,Female,no,no,Smartphone,private medical insurance,Phrase,LU7,Self,not_verified,LU
2,29156003,Contacted,0.0,46,Male,no,yes private,Smartphone,private health insurance,Broad,E14,Self,not_verified,E
3,26436791,No Contact,0.0,53,Female,no,no,Desktop,bupa health insurance,Exact,W2,Self + Partner,not_verified,W
4,80270348,Contacted,0.0,35,Male,no,no,Smartphone,private healthcare prices,Exact,BH23,Self,not_verified,BH


In [5]:
# Keyword grouping — brand/intent-aware categories
def categorize_keyword(kw):
    kw = str(kw).lower().strip()
    if "bupa" in kw:                                                    return "Brand: Bupa"
    elif any(b in kw for b in ["axa", "aviva", "saga", "vitality"]):    return "Brand: Other"
    elif any(w in kw for w in ["pre-existing", "preexisting"]):         return "Pre-existing Conditions"
    elif any(w in kw for w in ["price", "cost", "quote", "cheap"]):     return "Price / Quote Intent"
    elif any(w in kw for w in ["compare", "comparison", "best", "vs"]): return "Comparison / Research"
    elif any(w in kw for w in ["private health", "private medical"]):   return "Generic: Private Health"
    elif any(w in kw for w in ["health insurance", "medical insurance"]): return "Generic: Health Insurance"
    else: return "Other"

leads["keyword_group"] = leads["keyword"].apply(categorize_keyword)
print("KEYWORD GROUPS")
print(leads["keyword_group"].value_counts())

KEYWORD GROUPS
keyword_group
Generic: Private Health      3484
Brand: Bupa                  1594
Generic: Health Insurance    1219
Comparison / Research         488
Price / Quote Intent          419
Other                         192
Brand: Other                   89
Name: count, dtype: int64


In [6]:
# Target flags
leads["converted"] = (leads["lead_status"] == "Sold").astype(int)
leads["is_invalid"] = (leads["lead_status"] == "Invalid").astype(int)

# High value = top 20% premium among converted leads
premium_threshold = leads.loc[leads["converted"] == 1, "premium"].quantile(0.80)
leads["high_value"] = ((leads["converted"] == 1) & (leads["premium"] >= premium_threshold)).astype(int)

print(f"Conversion rate: {leads['converted'].mean():.2%}")
print(f"Invalid rate:    {leads['is_invalid'].mean():.2%}")
print(f"High-value threshold: £{premium_threshold:,.0f}")
print(f"High-value leads: {leads['high_value'].sum()}")

Conversion rate: 4.90%
Invalid rate:    4.36%
High-value threshold: £2,166
High-value leads: 74


In [7]:
# Age bands
leads["age_band"] = pd.cut(
    leads["age"],
    bins=[0, 25, 35, 45, 55, 65, 120],
    labels=["Under 25", "25-34", "35-44", "45-54", "55-64", "65+"]
)

# Synthetic dates (spread over 12 months, more leads on weekdays)
np.random.seed(42)
date_range = pd.date_range("2025-01-01", "2025-12-31", freq="D")
weights = np.where(date_range.weekday < 5, 1.3, 0.7)
weights = weights / weights.sum()
leads["created_date"] = np.random.choice(date_range, size=len(leads), p=weights)
leads = leads.sort_values("created_date").reset_index(drop=True)

print(f"Age bands: {leads['age_band'].value_counts().sort_index().to_dict()}")
print(f"Date range: {leads['created_date'].min().date()} → {leads['created_date'].max().date()}")

Age bands: {'Under 25': 523, '25-34': 1242, '35-44': 1997, '45-54': 1409, '55-64': 1131, '65+': 1183}
Date range: 2025-01-01 → 2025-12-31


In [8]:
# Quick quality check
print("QUALITY CHECK")
print("=" * 40)
print(f"Rows: {leads.shape[0]:,} | Cols: {leads.shape[1]}")
print(f"Duplicate IDs: {leads['lead_id'].duplicated().sum()}")
print(f"Age range: {leads['age'].min()} – {leads['age'].max()}")
print(f"Negative premiums: {(leads['premium'] < 0).sum()}")
print(f"\nNulls in key cols:")
for col in ["lead_id", "converted", "keyword_group", "device_type", "age"]:
    print(f"  {col}: {leads[col].isnull().sum()}")

QUALITY CHECK
Rows: 7,485 | Cols: 20
Duplicate IDs: 0
Age range: 20 – 119
Negative premiums: 0

Nulls in key cols:
  lead_id: 0
  converted: 0
  keyword_group: 0
  device_type: 0
  age: 0


## 2.5 · Validate the Cleaned Data with Pandera

Cleaning is only half the job — we also need a **safety net** that catches bad data before it leaks into the rest of the pipeline. That's where **Pandera** comes in.

**What is Pandera?** A lightweight library for declaring a **schema** for a pandas DataFrame: column types, allowed value ranges, allowed categories, nullability, uniqueness. You declare the schema once, then `schema.validate(df)` either passes silently or raises a clear error showing exactly which rows broke which rule.

**Why this matters in production:** in a real pipeline, the upstream CSV could change shape overnight (a new device type appears, a column gets renamed, ages become negative). Without a schema check, those problems silently corrupt your model training run and you only notice weeks later when the dashboard looks wrong. With a schema, the pipeline **fails loudly at the boundary** — exactly where you want failures to happen.

Below we declare a schema for the cleaned `leads` DataFrame and validate it. If anything is off, the cell will raise.

In [ ]:
import pandera.pandas as pa
from pandera import Column, Check

leads_schema = pa.DataFrameSchema(
    {
        "lead_id":             Column(int,   unique=True),
        "age":                 Column(int,   Check.in_range(18, 120)),
        "premium":             Column(float, Check.ge(0)),
        "gender":              Column(str,   Check.isin(["Male", "Female"])),
        "smoker":              Column(str,   Check.isin(["yes", "no"])),
        "device_type":         Column(str,   Check.isin(["Desktop", "Smartphone", "Tablet"])),
        "match_type":          Column(str,   Check.isin(["Exact", "Phrase", "Broad"])),
        "verification_status": Column(str,   Check.isin(["verified", "not_verified"])),
        "keyword_group":       Column(str),
        "pc_area":             Column(str,   nullable=True),
        "converted":           Column(int,   Check.isin([0, 1])),
        "is_invalid":          Column(int,   Check.isin([0, 1])),
        "high_value":          Column(int,   Check.isin([0, 1])),
    },
    strict=False,  # allow extra columns we don't validate
)

leads_validated = leads_schema.validate(leads, lazy=True)
print(f"✅ Schema valid — {len(leads_validated):,} rows passed all checks.")

**How to read this:**

- `lazy=True` collects **all** failures before raising, instead of stopping at the first bad row. In a real pipeline you want the full picture, not just the first error.
- If a check fails, Pandera raises a `SchemaErrors` exception with a table of which column failed which rule and how many rows were affected — perfect for a CI log or a Slack alert.
- We set `strict=False` so the schema only validates the columns we listed; extra columns (like `created_date`, `week_start`) pass through untouched.

**Where this fits later:** when we move this notebook into `src/data/validate.py`, this exact schema becomes a reusable function the pipeline calls right after cleaning. Fail the run if validation fails — don't let bad data reach the model.

## 3 · Build Mart Table

Aggregate leads weekly by `keyword_group × device × match_type`, then join to spend data for KPIs.

In [9]:
leads["week_start"] = leads["created_date"].dt.to_period("W").apply(lambda x: x.start_time)

lead_weekly = (
    leads.groupby(["week_start", "keyword_group", "device_type", "match_type"])
    .agg(
        total_leads=("lead_id", "count"),
        conversions=("converted", "sum"),
        invalids=("is_invalid", "sum"),
        total_premium=("premium", "sum"),
        high_value_leads=("high_value", "sum"),
    )
    .reset_index()
)

lead_mart = lead_weekly.merge(
    lead_spend,
    on=["week_start", "keyword_group", "device_type", "match_type"],
    how="outer",
).fillna(0)

# KPIs
lead_mart["conversion_rate"] = (lead_mart["conversions"] / lead_mart["total_leads"].replace(0, np.nan)).round(4)
lead_mart["cac"] = (lead_mart["spend"] / lead_mart["conversions"].replace(0, np.nan)).round(2)
lead_mart["roas"] = (lead_mart["total_premium"] / lead_mart["spend"].replace(0, np.nan)).round(2)
lead_mart["rpl"] = (lead_mart["total_premium"] / lead_mart["total_leads"].replace(0, np.nan)).round(2)
lead_mart["net_per_lead"] = (lead_mart["rpl"] - COST_PER_LEAD).round(2)

print(f"Lead mart: {lead_mart.shape}")
print(f"Overall RPL: £{lead_mart['total_premium'].sum() / lead_mart['total_leads'].sum():.2f}")
lead_mart.head()

Lead mart: (2875, 18)
Overall RPL: £76.66


,week_start,keyword_group,device_type,match_type,total_leads,conversions,invalids,total_premium,high_value_leads,impressions,clicks,spend,avg_cpc,conversion_rate,cac,roas,rpl,net_per_lead
0,2024-12-30,Brand: Bupa,Desktop,Exact,2.0,1.0,0.0,2182.20,1.0,0.0,0.0,0.0,0.0,0.50,0.0,NaN,1091.10,1036.10
1,2024-12-30,Brand: Bupa,Smartphone,Exact,12.0,3.0,0.0,3472.92,0.0,0.0,0.0,0.0,0.0,0.25,0.0,NaN,289.41,234.41
2,2024-12-30,Comparison / Research,Desktop,Broad,1.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,NaN,NaN,0.00,-55.00
3,2024-12-30,Comparison / Research,Smartphone,Broad,2.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,NaN,NaN,0.00,-55.00
4,2024-12-30,Comparison / Research,Smartphone,Exact,1.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,NaN,NaN,0.00,-55.00


In [10]:
## 4 · Save Everything
# Staging
leads.to_parquet(STAGING_DIR / "leads_clean.parquet", index=False)

In [11]:
# Mart
lead_mart.to_parquet(MART_DIR / "lead_performance.parquet", index=False)

print("Saved to staging/:")
print(f"  leads_clean.parquet          ({leads.shape[0]:,} rows)")
print(f"\nSaved to mart/:")
print(f"  lead_performance.parquet     ({lead_mart.shape[0]:,} rows)")

Saved to staging/:
  leads_clean.parquet          (7,485 rows)

Saved to mart/:
  lead_performance.parquet     (2,875 rows)
